# Matrix transpose on a T4: shared memory, tiling and bank conflicts

Companion to the card *Shared memory, tiling and bank conflicts* (perf-3-kernels/shared-memory-transpose.html).

**Runtime → Change runtime type → T4 GPU**, then run all cells.

You will build and time the five kernels from Mark Harris, ["An Efficient Matrix Transpose in CUDA C/C++"](https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/), on a 1024×1024 float matrix:
`copy`, `copySharedMem`, `transposeNaive`, `transposeCoalesced` and `transposeNoBankConflicts`.
Each result is checked against a CPU transpose, then printed as effective bandwidth (2 × 4 MiB ÷ time), as a % of `copy`, and as a % of the T4's 320 GB/s peak.

Colab's T4 clocks and availability vary between sessions, so run it twice and expect some spread.

In [ ]:
!nvidia-smi

In [ ]:
%%writefile transpose.cu
// The five kernels from Mark Harris, "An Efficient Matrix Transpose in CUDA C/C++"
// (NVIDIA blog), kept as in the post. Host code: every CUDA call checked,
// warm-up launch, cudaEvent timing over NUM_REPS launches, CPU reference check.
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call) do { cudaError_t e_ = (call); if (e_ != cudaSuccess) { \
  fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(e_), __FILE__, __LINE__); \
  exit(1); } } while (0)

const int TILE_DIM = 32;
const int BLOCK_ROWS = 8;
const int NUM_REPS = 100;
const double T4_PEAK_GBS = 320.0;

__global__ void copy(float *odata, const float *idata) {
  int x = blockIdx.x * TILE_DIM + threadIdx.x;
  int y = blockIdx.y * TILE_DIM + threadIdx.y;
  int width = gridDim.x * TILE_DIM;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    odata[(y + j) * width + x] = idata[(y + j) * width + x];
}

__global__ void copySharedMem(float *odata, const float *idata) {
  __shared__ float tile[TILE_DIM * TILE_DIM];
  int x = blockIdx.x * TILE_DIM + threadIdx.x;
  int y = blockIdx.y * TILE_DIM + threadIdx.y;
  int width = gridDim.x * TILE_DIM;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    tile[(threadIdx.y + j) * TILE_DIM + threadIdx.x] = idata[(y + j) * width + x];
  __syncthreads();  // not needed for correctness here; kept to mimic the transpose
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    odata[(y + j) * width + x] = tile[(threadIdx.y + j) * TILE_DIM + threadIdx.x];
}

__global__ void transposeNaive(float *odata, const float *idata) {
  int x = blockIdx.x * TILE_DIM + threadIdx.x;
  int y = blockIdx.y * TILE_DIM + threadIdx.y;
  int width = gridDim.x * TILE_DIM;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    odata[x * width + (y + j)] = idata[(y + j) * width + x];
}

__global__ void transposeCoalesced(float *odata, const float *idata) {
  __shared__ float tile[TILE_DIM][TILE_DIM];
  int x = blockIdx.x * TILE_DIM + threadIdx.x;
  int y = blockIdx.y * TILE_DIM + threadIdx.y;
  int width = gridDim.x * TILE_DIM;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    tile[threadIdx.y + j][threadIdx.x] = idata[(y + j) * width + x];
  __syncthreads();
  x = blockIdx.y * TILE_DIM + threadIdx.x;  // transpose block offset
  y = blockIdx.x * TILE_DIM + threadIdx.y;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    odata[(y + j) * width + x] = tile[threadIdx.x][threadIdx.y + j];
}

__global__ void transposeNoBankConflicts(float *odata, const float *idata) {
  __shared__ float tile[TILE_DIM][TILE_DIM + 1];
  int x = blockIdx.x * TILE_DIM + threadIdx.x;
  int y = blockIdx.y * TILE_DIM + threadIdx.y;
  int width = gridDim.x * TILE_DIM;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    tile[threadIdx.y + j][threadIdx.x] = idata[(y + j) * width + x];
  __syncthreads();
  x = blockIdx.y * TILE_DIM + threadIdx.x;
  y = blockIdx.x * TILE_DIM + threadIdx.y;
  for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS)
    odata[(y + j) * width + x] = tile[threadIdx.x][threadIdx.y + j];
}

typedef void (*Kernel)(float *, const float *);

// Returns effective bandwidth in GB/s (2 x matrix bytes / time), or -1 if the result is wrong.
double run(const char *name, Kernel k, dim3 grid, dim3 block, float *d_out, const float *d_in,
           float *h_out, const float *ref, int n, cudaEvent_t start, cudaEvent_t stop) {
  size_t bytes = (size_t)n * sizeof(float);
  CUDA_CHECK(cudaMemset(d_out, 0, bytes));
  for (int w = 0; w < 3; w++) k<<<grid, block>>>(d_out, d_in);  // warm-up
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaDeviceSynchronize());
  CUDA_CHECK(cudaEventRecord(start));
  for (int r = 0; r < NUM_REPS; r++) k<<<grid, block>>>(d_out, d_in);
  CUDA_CHECK(cudaEventRecord(stop));
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaEventSynchronize(stop));
  float ms = 0;
  CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
  CUDA_CHECK(cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost));
  for (int i = 0; i < n; i++)
    if (h_out[i] != ref[i]) {
      printf("%-26s FAILED at %d: got %f want %f\n", name, i, h_out[i], ref[i]);
      return -1;
    }
  double per_launch_s = ms * 1e-3 / NUM_REPS;
  return 2.0 * bytes / per_launch_s / 1e9;
}

int main() {
  const int nx = 1024, ny = 1024, n = nx * ny;
  const size_t bytes = (size_t)n * sizeof(float);
  dim3 grid(nx / TILE_DIM, ny / TILE_DIM), block(TILE_DIM, BLOCK_ROWS);

  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  printf("Device: %s (sm_%d%d, %d SMs, %zu KB shared memory per SM)\n", prop.name, prop.major,
         prop.minor, prop.multiProcessorCount, prop.sharedMemPerMultiprocessor / 1024);
  printf("Matrix %dx%d floats, block %dx%d threads, tile %dx%d, %d reps\n\n", nx, ny, TILE_DIM,
         BLOCK_ROWS, TILE_DIM, TILE_DIM, NUM_REPS);

  float *h_in = (float *)malloc(bytes), *h_out = (float *)malloc(bytes), *gold = (float *)malloc(bytes);
  for (int j = 0; j < ny; j++)
    for (int i = 0; i < nx; i++) h_in[j * nx + i] = (float)(j * nx + i);
  for (int j = 0; j < ny; j++)
    for (int i = 0; i < nx; i++) gold[j * nx + i] = h_in[i * nx + j];

  float *d_in, *d_out;
  CUDA_CHECK(cudaMalloc(&d_in, bytes));
  CUDA_CHECK(cudaMalloc(&d_out, bytes));
  CUDA_CHECK(cudaMemcpy(d_in, h_in, bytes, cudaMemcpyHostToDevice));
  cudaEvent_t start, stop;
  CUDA_CHECK(cudaEventCreate(&start));
  CUDA_CHECK(cudaEventCreate(&stop));

  struct { const char *name; Kernel k; const float *ref; } ks[] = {
    {"copy", copy, h_in},
    {"copySharedMem", copySharedMem, h_in},
    {"transposeNaive", transposeNaive, gold},
    {"transposeCoalesced", transposeCoalesced, gold},
    {"transposeNoBankConflicts", transposeNoBankConflicts, gold},
  };
  double copy_gbs = 0;
  printf("%-26s %10s %10s %12s\n", "kernel", "GB/s", "% of copy", "% of 320 GB/s");
  for (int i = 0; i < 5; i++) {
    double gbs = run(ks[i].name, ks[i].k, grid, block, d_out, d_in, h_out, ks[i].ref, n, start, stop);
    if (gbs < 0) continue;
    if (i == 0) copy_gbs = gbs;
    printf("%-26s %10.1f %9.1f%% %11.1f%%\n", ks[i].name, gbs, 100.0 * gbs / copy_gbs,
           100.0 * gbs / T4_PEAK_GBS);
  }

  CUDA_CHECK(cudaEventDestroy(start));
  CUDA_CHECK(cudaEventDestroy(stop));
  CUDA_CHECK(cudaFree(d_in));
  CUDA_CHECK(cudaFree(d_out));
  free(h_in); free(h_out); free(gold);
  return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o transpose transpose.cu && ./transpose

## Compare with the blog (reference numbers, not a T4)

Harris's table, effective bandwidth in GB/s with ECC on:

| Kernel | Tesla M2050 | Tesla K20c |
|---|---|---|
| copy | 105.2 | 136.0 |
| copySharedMem | 104.6 | 152.3 |
| transposeNaive | 18.8 | 55.3 |
| transposeCoalesced | 51.3 | 97.6 |
| transposeNoBankConflicts | 99.5 | 144.3 |

Step multipliers from that table (our recomputation): naive → coalesced 2.73× / 1.76×, coalesced → padded 1.94× / 1.48×, padded = 94.6% / 94.7% of the faster copy.

Questions for your run: does the ordering hold on Turing? How big is the naive → coalesced step, and how big is the padding step? Newer GPUs cache global memory more aggressively, so the naive kernel's penalty may be smaller than on the M2050.

## Try this
1. Change `TILE_DIM + 1` to `TILE_DIM + 2` in `transposeNoBankConflicts`. Why does an even pad bring conflicts back (2-way)?
2. Swap the padding for an XOR swizzle: declare `tile[TILE_DIM][TILE_DIM]`, store `tile[r][c ^ r]`, and read `tile[threadIdx.x][(threadIdx.y + j) ^ threadIdx.x]`.
3. Profile the bank conflicts directly if Nsight Compute is available: `!ncu --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum ./transpose` (Colab may not allow profiler access).